In [ ]:
import psycopg2 as psy
import pandas as pd
from sqlalchemy import create_engine
from transform import transformed_data



def load_to_postgres(transformed_data):

    engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")

    df = pd.DataFrame([transformed_data])

    try:
    
        df.to_sql("weather_report", con=engine, if_exists="replace", index=False)

        print("\n[SUCCESS] Data successfully pushed to SQL table 'weather_reports'")
        
        with engine.connect() as conn:

            result = pd.read_sql("SELECT * FROM weather_report", conn)
            
            print("\n--- Last Row in Database ---")

            print(result)

        return True
    
    except Exception as e:
        
        print(f"Postgres Load Error :{e}")

        return False


{'coord': {'lon': 10.99, 'lat': 44.34}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 292.12, 'feels_like': 291.72, 'temp_min': 292.08, 'temp_max': 292.74, 'pressure': 1017, 'humidity': 63, 'sea_level': 1017, 'grnd_level': 951}, 'visibility': 10000, 'wind': {'speed': 3.22, 'deg': 62, 'gust': 3.02}, 'clouds': {'all': 58}, 'dt': 1776436920, 'sys': {'type': 2, 'id': 2075663, 'country': 'IT', 'sunrise': 1776400148, 'sunset': 1776448904}, 'timezone': 7200, 'id': 3163858, 'name': 'Zocca', 'cod': 200}
    City Country      Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  broken clouds   18.97         18.57        63


In [4]:
df = pd.DataFrame([transformed_data])


In [ ]:
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")


df.to_sql("weather_report", con=engine, if_exists="replace", index=False)

print("\n[SUCCESS] Data successfully pushed to SQL table 'weather_reports'")


[SUCCESS] Data successfully pushed to SQL table 'weather_reports'


In [10]:
with engine.connect() as conn:

            result = pd.read_sql("SELECT * FROM weather_report ", conn)
            
            print("\n--- Last Row in Database ---")

            print(result)


--- Last Row in Database ---
    City Country      Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  broken clouds   18.97         18.57        63


In [23]:
import os
from dotenv import load_dotenv

load_dotenv()


host     = os.getenv("DB_HOST")
port     = os.getenv("DB_PORT")
user     = os.getenv("DB_USER")
dbname   = os.getenv("DB_NAME")
password = os.getenv("DB_PASSWORD")
sslmode  = "require"


engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")


In [27]:
from extract import extract_weather
from transform import transform_weather
from load import load_to_postgres
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os 

def run_pipeline ():
    print("--- Starting Weather API ETL Pipeline ---")

    # STAGE 1: EXTRACT
    print("[1/3] Extracting data from OpenWeather...")

    API_KEY = os.getenv("API_KEY")
    LAT     = os.getenv("LAT")
    LON     = os.getenv("LON")

    raw_data = extract_weather(LAT, LON, API_KEY)
    
    if not raw_data or raw_data.get("cod") != 200:
        print("ERROR: Extraction failed. Check API key or coordinates.")
        return

    # STAGE 2: TRANSFORM
    print("[2/3] Transforming data (Cleaning & Unit Conversion)...")
    clean_data = transform_weather()
    
    # STAGE 3: LOAD
    print(f"[3/3] Loading data for {clean_data['City']} into Database...")

    success = load_to_postgres(clean_data)
   

    if success:

        print("--- Pipeline Completed Successfully! ---")
    else:
        print("--- Pipeline Failed at Load Stage ---")

if __name__ == "__main__":
    run_pipeline()

--- Starting Weather API ETL Pipeline ---
[1/3] Extracting data from OpenWeather...
[2/3] Transforming data (Cleaning & Unit Conversion)...
[3/3] Loading data for Zocca into Database...

[SUCCESS] Data successfully pushed to SQL table 'weather_report'

--- Last Row in Database ---
    City Country      Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  broken clouds   18.97         18.57        63
--- Pipeline Completed Successfully! ---


In [28]:
from extract import extract_weather
from transform import transform_weather
from load import load_to_postgres
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os 

def run_pipeline ():
    print("--- Starting Weather API ETL Pipeline ---")

    # STAGE 1: EXTRACT
    print("[1/3] Extracting data from OpenWeather...")

    API_KEY = os.getenv("API_KEY")
    LAT     = os.getenv("LAT")
    LON     = os.getenv("LON")

    raw_data = extract_weather(LAT, LON, API_KEY)
    
    if not raw_data or raw_data.get("cod") != 200:
        print("ERROR: Extraction failed. Check API key or coordinates.")
        return

    # STAGE 2: TRANSFORM
    print("[2/3] Transforming data (Cleaning & Unit Conversion)...")
    clean_data = transform_weather()
    
    # STAGE 3: LOAD

    host     = os.getenv("DB_HOST")
    port     = os.getenv("DB_PORT")
    user     = os.getenv("DB_USER")
    dbname   = os.getenv("DB_NAME")
    password = os.getenv("DB_PASSWORD")
    sslmode  = "require"

    print(f"[3/3] Loading data for {clean_data['City']} into Database...")

    success = load_to_postgres(clean_data)
   

    if success:

        print("--- Pipeline Completed Successfully! ---")
    else:
        print("--- Pipeline Failed at Load Stage ---")

if __name__ == "__main__":
    run_pipeline()

--- Starting Weather API ETL Pipeline ---
[1/3] Extracting data from OpenWeather...
[2/3] Transforming data (Cleaning & Unit Conversion)...
[3/3] Loading data for Zocca into Database...

[SUCCESS] Data successfully pushed to SQL table 'weather_report'

--- Last Row in Database ---
    City Country      Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  broken clouds   18.97         18.57        63
--- Pipeline Completed Successfully! ---
